<img src="images/banner.png" style="width: 100%;">

# Convolutional Neural Networks (CNN)

**MSDS 2026 | COSCI223: Machine Learning 3**

**Convolutional networks (ConvNets)** or **convolutional neural networks (CNNs)** are a family of models that were inspired by how the visual cortex of human brain works when recognizing objects.

A ConvNet employs a mathematical operation called **convolution**, a specialized kind of linear operation. Accordingly, ConvNets are simply neural networks that use convolution in at least one of their layers.

Convolution layers learn local patterns, giving CNNs two interesting properties:

1. *The patterns they learn are **translation invariant**,* making ConvNets data efficient when processing images becasue the visual world is fundamentally translation invaariant.
    - They need fewer training samples to learn representations that have generalization power.
2. *They can learn **spatial hierarchies** of patterns,* allowing ConvNets efficiently learn increasingly complex and abstract visual concepts because the visual world is fundamentally spatially hierarchical.

<img src="images/spatial-hierarchy.png" style="width: 100%;">
Source: Deep Learning with Python (Chollet & Chollet, 2026)

CNN is the type of deep learning model that is used by most computer vision applications.

- First Application
    > [Backpropagation Applied to Handwritten Zip Code Recognition](http://yann.lecun.com/exdb/publis/pdf/lecun-89e.pdf) (LeCun et al., 1989)


- Current Applications
    1. **Classification** of images, sometimes called *image recognition*
    2. **Detection** of objects in an image and determining their locations within the image
    3. **Segmentation** of images, in which each pixel is classified individually thereby dividing the image into regions sharing a common label
    4. **Caption generation** in which a textual description is generated automatically from an image
    5. **Synthesis** of new images, such as image generation of synthesizing based on a text input describing the desired image content
    6. **Inpainting** in which a region of an image is replaced with synthesized pixels that are consistent with the rest of the image. 
    7. **Style transfer** in which an input image in one style, for example a photograph, is transformed into a corresponding image in a different style, for example an oil painting.
    8. **Super-resolution** in which the resolution of an image is improved by increasing the number of pixels and synthesizing associated high-frequency information.
    9. **Depth prediction** in which one or more views are used to predict the distance of the scene from the camera at each pixel in a target image.
    10. **Scene reconstruction** in which one or more two-dimensional images of a scene are used to reconstruct a three-dimensional representation.

In [ ]:
import keras
from keras import layers
from keras.models import Sequential
from keras import models

import numpy as np
import pandas as pd
from matplotlib import rcParams
import matplotlib.pyplot as plt


# Some preambles for prettification
rcParams.update({'figure.figsize': (8, 6), 'axes.spines.top': False,
                 'axes.spines.right': False, 'axes.labelsize': 12,
                 'axes.titlesize': 12, 'axes.titleweight': 'bold',
                 'lines.linewidth': 3})

# Building Blocks of CNNs

As you can see in the following image, a CNN computes feature maps from an input image, where each element comes from a local patch of pixels in the input image:

<img width="919" alt="CNN_dog" src="https://user-images.githubusercontent.com/25600601/134186781-c3eb2e95-fcce-4f65-8f86-60317bfa11d7.png">


CNNs perform very well for image-related tasks, and that's largely due to two important ideas:     

- **Sparse-connectivity:** A single element in the feature map is connected to only a small patch of pixels. (This is very different from connecting to the whole input image, in the case of perceptrons. You may find it useful to look back and compare how we implemented a fully connected network that connected to the whole image.)    

- **Parameter-sharing:** The same weights are used for different patches of the input image. As a direct consequence of these two ideas, the number of weights (parameters) in the network decreases dramatically, and we see an improvement in the ability to capture salient features. Intuitively, it makes sense that *nearby pixels are probably more relevant to each other* than pixels that are far away from each other.

Typically, CNNs are composed of *several Convolution (Conv)* layers and *subsampling (also known as Pooling (P))* layers that are followed by one or more Fully Connected (FC) layers at the end. The fully connected layers are essentially a multilayer perceptron, where every input unit $i$ is connected to every output unit $j$ with weight $w_{ij}$.

## The convolution operation

The **`convolution operation`** extracts patches from its `input feature map` and applies the same transformation to all of these patches, producing an `output feature map`.

These **feature maps** have two spatial axes (`height` and `width`) as well as a `depth` axis (also called the `channels` axis). 
- `input depth`: 1 for black-and-white images, 3 for an RGB image
- `output depth`: can be arbitrary because the output depth is a parameter of the layer, and the different channels in that depth axis stand for *`filters`*
- `filters`: encode specific aspects of the input data
- `response map` of the filter over the input, indicating the response of a filter pattern at different locations in the input

Convolutions are defined by **two key parameters**:

1. *Size of the patches (`window_height`, `window_width`) extracted from the inputs*, typically $3x3$ or $5x5$
2. *Depth of the output feature map*,  the number of filters computed by the convolution

A convolution works by sliding these windows over the input feature map (`height`, `width`, `input_depth`), stopping at every possible location, and extracting the patch of surrounding features of shape (`window_height`, `window_width`, `input_depth`). Each patch is then transformed into a 1D vector of shape (`output_depth`), which is done via a tensor product with a learned weight matrix, called **convolution kernel**. The same kernel is reused across every patch. 

All of these vectors (one per patch) are then spatially reassembled into an output feature map (`height`, `width`, `output_depth`). Every spatial location in the output feature map corresponds to the same location in the input feature map.

Note that the output width and height may differ from the input width and height for two reasons:
- boarder effects
- use of strides

#### Padding (p)
- to pad the edges of the inputs with new values and proceed as usual to retain the input shape

#### Stride (s)
- the distance between two successive windows

### The size of the convolution output

The output size of a convolution is determined by the total  number of times that we shift the filter  $\mathbf{w}$  along the input vector. Again, if the input vector $\mathbf{x}$ has size
$n$ and the filter  $\mathbf{w}$ is of size $m$. Then, the size of the output resulting from  $\mathbf{x} *  \mathbf{w}$  with padding $p$ and stride $s$ is determined as follows:

\begin{equation}
o=floor (\frac{n + 2p - m}{s}) + 1
\end{equation}

Here, the floor operation returns the largest integer that is equal or smaller to the input, e.g. floor (1.43) =1



Consider the following example:

  • Compute the output size for an input vector of size 10 with a convolution kernel of size 5, padding 2, and stride 1:
    
\begin{equation}
n=10, m=5, p=2, s=1, \Rightarrow o= floor(\frac{10+ 2 \times 2 -5}{1}) + 1 = 10
\end{equation}

(Note that in this case, the output size turns out to be the same as the input; therefore, we conclude this as **mode='same' **)

See illustration below if you want to trace it perfectly:

![CNN_Convolution_Example1](https://user-images.githubusercontent.com/25600601/134187810-ed86251d-ddef-45d0-ae1d-62ff80c7d56b.png)


### Performing discrete convolutions in 1D

Let's start with some basic definitions and notations we are going to use. A discrete convolution for two one-dimensional vectors $\mathbf{x}$ and $\mathbf{w}$ is denoted by $\mathbf{y}= \mathbf{x}*\mathbf{w}$ , in which vector $\mathbf{x}$ is our input (sometimes called signal) and $\mathbf{w}$ is called the filter or kernel. A discrete convolution is mathematically defined as follows:   

\begin{equation}
\mathbf{y}= \mathbf{x} * \mathbf{w} = \sum_{k=-\infty}^{k=+\infty} \mathbf{x} [i-k] \mathbf{w}[k]
\end{equation}

Here, the brackets [...] are used to denote the indexing for vector elements. The index $i$ runs through each element of the output vector $\mathbf{y}$. There are two odd things in the preceding formula that we need to clarify: $−\infty$ to $+\infty$ indices and negative indexing for $\mathbf{x}$.

The **first issue** where the **sum runs through indices from $−\infty$ to $+\infty$** seems odd mainly because in machine learning applications, we always deal with finite feature vectors. For example, if $\mathbf{x}$ has 10 features with indices 0,1,2,…,8,9, then indices $−\infty:-1$ and $10 : +\infty$ are out of bounds for $\mathbf{x}$. Therefore, to correctly compute the summation shown in the preceding formula, it is assumed that $\mathbf{x}$ and $\mathbf{w}$ are filled with zeros. This will result in an output vector $\mathbf{y}$ that also has infinite size with lots of zeros as well. Since this is not useful in practical situations, $\mathbf{x}$ is padded only with a
finite number of zeros.   

This process is called **zero-padding** or simply **padding**. Here, the number of zeros padded on each side is denoted by *p*. An example padding of a one-dimensional vector $\mathbf{x}$ is shown in the following figure:

<img width="670" alt="zero-padding" src="https://user-images.githubusercontent.com/25600601/134187112-666edbe5-6e2f-4982-b01b-721ae3f53969.png">


Let's assume that the original input $\mathbf{x}$ and filter $\mathbf{w}$ have $n$ and $m$ elements, respectively, where $m \leq n$. Therefore, the padded vector $\mathbf{x}^p$  has size $n + 2p$. Then, the practical formula for computing a discrete convolution will change to the following:


\begin{equation}
\mathbf{y}= \mathbf{x} * \mathbf{w} \Rightarrow \mathbf{y}[i]= \sum_{k=0}^{k=m-1} \mathbf{x}^p [i+m-k] \mathbf{w}[k]
\end{equation}


Now that we have solved the infinite index issue, the second issue is indexing $\mathbf{x}$ with $i + m - k$. The important point to notice here is that $\mathbf{x}$ and $\mathbf{w}$ are indexed in different directions in this summation. For this reason, we can flip one of those vectors, $\mathbf{x}$ or $\mathbf{w}$, after they are padded. Then, we can simply compute their dot product.    

Let's assume we flip the filter $\mathbf{w}$ to get the rotated filter $\mathbf{w}^r$ . Then, the dot product $\mathbf{x}[i : i + m]$ $\cdot$ $\mathbf{w}^r$ is computed to get one element $\mathbf{y}[i]$, where $\mathbf{x}[i : i + m]$ is a patch of $\mathbf{x}$ with size $m$.

This operation is repeated like in a sliding window approach to get all the output elements. The following figure provides an example with $\mathbf{x} = (3,2,1,7,1,2,5,4)$ and $\mathbf{w}$ = ($\frac{1}{2}, \frac{3}{4}, 1, \frac{1}{4}$) so that the first three output elements are computed:

<img width="738" alt="padding_illustration" src="https://user-images.githubusercontent.com/25600601/134187244-781c5e5e-064a-4194-8ea8-be13285e41fb.png">

You can see in the preceding example that the padding size is zero ($p = 0$). Notice that the rotated filter wr is shifted by two cells each time we shift. This shift is another hyperparameter of a convolution, the stride $s$. In this example, the **stride is two**, $s = 2$. Note that the stride has to be a positive number smaller than the size of the input vector $\mathbf{x}$ (pause and answer why?). We'll talk more about padding and strides in the next section!

#### Exploration Task 1

In order to learn how to compute convolutions in one dimension, a naive implementation is shown in the following code block, and the results are compared with the numpy.convolve function. The code is given below:

In [ ]:
def conv1d(x, w, p, s=1):
    w_rot = np.array(w[::-1])
    x_padded = np.array(x)
    if p > 0:
        zero_pad = np.zeros(shape=p)
        x_padded = np.concatenate([zero_pad, x_padded, zero_pad])
    res = []
    for i in range(0, int(len(x)/s),s):
        res.append(np.sum(x_padded[i:i+w_rot.shape[0]] * w_rot))
    return np.array(res)

**Example**

In [ ]:
x = None
w = None

In [ ]:
print('Conv1d Implementation:', conv1d(x, w, p=2, s=1))

In [ ]:
print('Numpy Results:', np.convolve(x, w, mode='same'))

<div class="alert alert-block alert-info"> 
<b>INSTRUCTION</b>
<br>
Create a 1-D array and a filter. Perform the convolution operation.
<br>
</div>

### Performing discrete convolutions in 2D

The concepts you learned in the previous sections are easily extendible to two dimensions. When we deal with two dimensional input, such as a matrix $\mathbf{X}_{n1×n2}$ and the filter matrix $\mathbf{W}_{m1×m2}$, where $m_1 \leq n_1$ and $m_2 \leq n_2$ , then the matrix $\mathbf{Y}$ = $\mathbf{X} * \mathbf{W}$ is the result of 2D convolution of $\mathbf{X}$ with $\mathbf{W}$. This is mathematically defined as follows:


\begin{equation}
\mathbf{Y}= \mathbf{X} * \mathbf{W} \Rightarrow \mathbf{Y}[i, j]= \sum_{k_1=-\infty}^{k_2=\infty}\sum_{k_2=-\infty}^{\infty} \mathbf{X} [i-k_1, j-k_2] \mathbf{W}[k_1, k_2]
\end{equation}

Notice that if you omit one of the dimensions, the remaining formula is exactly the same as the one we used previously to compute the convolution in 1D. In fact, all the previously mentioned techniques, such as zero-padding, rotating the filter matrix, and the use of strides, are also applicable to 2D convolutions, provided that they are extended to both the dimensions independently. The following example illustrates the computation of a 2D convolution between an input matrix $\mathbf{X}_{3×3}$, a kernel matrix $\mathbf{W}_{3×3}$, padding $p = (1, 1)$, and stride $s = (2, 2)$. According to the specified padding, one layer of zeros are padded on each side of the input matrix, which results in the padded matrix $\mathbf{X}_{5×5}^{padded}$, as follows:

<img width="746" alt="padded_2d" src="https://user-images.githubusercontent.com/25600601/134188176-c6a412b6-c6bd-49a4-bbdd-508b285e0882.png">

With the preceding filter, the rotated filter will be:

<img width="373" alt="Wrotated" src="https://user-images.githubusercontent.com/25600601/134188347-20df516d-0292-430a-91d5-bcc9eebc366e.png">

Note that this rotation is not the same as the transpose matrix. To get the rotated filter in NumPy, we can write W_rot=W[::-1,::-1]. Next, we can shift the rotated filter matrix along the padded input matrix $\mathbf{X}$ padded like a sliding window and compute the sum of the element-wise product, which is denoted by the dot operator in the
following figure:

<img width="900" alt="2d_convolution" src="https://user-images.githubusercontent.com/25600601/134188450-7da6a7f4-cea5-4b0d-80cd-34b56d3804ee.png">

The result will be the $2 \times 2$ matrix $\mathbf{Y}$.    

## The CNN architecture

A model’s “architecture” is the sum of the choices that went into creating it: which layers to use, how to configure them, in what arrangement to connect them. 

<img width="704" alt="CNN_full" src="https://user-images.githubusercontent.com/25600601/134190140-22451833-879f-4696-b365-7eb22dcd3933.png">

### The modularity-hierarchy-reuse formula

The CNN model should be organized into repeated **`blocks`** of layers, usually made of multiple convolution layers and a max pooling layer. The number of filters in the layers should increase as the size of the spatial feature maps decreases.
- The width and height dimensions tend to shrink as you go deeper in the model.
- The number of channels is controlled by the first argument passed to the convulation layer.

This formula underpins the architecture of nearly all complex systems, including deep learning models. 

### Convolution layers
- **`Conv1D`**: [1D convolution layer](https://keras.io/api/layers/convolution_layers/convolution1d/)
    - sequential signals or text data
- **`Conv2D`**:[2D convolution layer](https://keras.io/api/layers/convolution_layers/convolution2d/)
    - images, time-frequency representations (speech and audio)
- **`Conv3D`**:[3D convolution layer](https://keras.io/api/layers/convolution_layers/convolution3d/)
- [Other convolution layers](https://keras.io/api/layers/convolution_layers/)
    - video, volumetric images, tomography images

#### Sample Dataset

We will use the MNIST dataset in this notebook, and in the associated exercise.

In [ ]:
# Load dataset
(x_trainval, y_trainval), (x_test, y_test) = keras.datasets.mnist.load_data()

In [ ]:
# Check the shape of the input and output

In [ ]:
# Plot class distribution

**Example**
  
Here, we apply a 2D convolution layer with one (1) 3x3 filter to a sample image from the MNIST dataset, visualize the feature maps, the **convolution kernel** (learned weight matrix).

In [ ]:
# Display a sample image

In [ ]:
# Normalize

In [ ]:
# Check the dimensions of the input

Note that the **`input_shape`** in Keras follows the following syntax: 
- (`batch_size`, `height`, `width`, `channels`)

In [ ]:
# Reshape to (1, 28, 28, 1) - batch size, height, width, channels

In [ ]:
# Define a Conv2D layer with 1 3x3 filter

In [ ]:
# Display input feature map

In [ ]:
# Get the weights from the Conv2D layer

- weights shape → (3, 3, 1, 1)
    - kernel height = 3
    - kernel width = 3
    - input channels = 1
    - number of filters = 1
- bias shape → (1,)

In [ ]:
# Get the actual kernel values

In [ ]:
# Display the convolution kernel

#### Exploration Task 2

<div class="alert alert-block alert-info"> 
<b>INSTRUCTIONS</b>
<br>
Apply a 2D convolution layer with <b>four (4)</b> 3x3 filters to the same sample image (stored as <b>`img`</b>), visualize the feature maps, and the convolution kernels. 
<br>1. Define a Conv2D layer with 4 3x3 filters
<br>2. Apply a convolution 
<br>3. Display the input feature maps
<br>4. Get the weights from the Conv2D layer
<br>5. Visualize the four convolution kernels
<br>6. Adding an activation layer
<br>7. Visualize the four output feature maps
<br>
</div>

In [ ]:
# Define a Conv2D layer with 4 3x3 filters

In [ ]:
# Apply a convolution

In [ ]:
# Display the input feature maps

In [ ]:
# Get the weights from the Conv2D layer

In [ ]:
# Visualize the four convolution kernels

In [ ]:
# Visualize the four output feature maps

### Pooling layer

This replaces the output of the net at a certain location with a summary statistic of the nearby outputs.

- https://keras.io/api/layers/pooling_layers/

### Other Layers

#### Batch Normalization

This applies a transformation that maintains the mean output close to 0 and the output standard deviation close to 1. 

- https://keras.io/api/layers/normalization_layers/batch_normalization/

#### Dropout Layer

This randomly sets input units to 0 with a frequency of rate at each step during training time, which helps prevent overfitting.

- https://keras.io/api/layers/regularization_layers/dropout/

#### Fully connected layers
These layers are stacks of dense layers that serve as the classifier.
- https://keras.io/api/layers/core_layers/dense/

### Principles

- Your model should be organized into repeated **`blocks`** of layers, usually made of multiple convolution layers and a max pooling layer.
- The number of filters in your layers should increase as the size of the spatial feature maps decreases.
- Deep and narrow is better than broad and shallow.
- By default each feature map has a bias inherently added in Keras.
- Introducing *`residual connections`* around blocks of layers helps you train deeper networks. 
- It can be beneficial to introduce *`batch normalization`* layers after your convolution layers.
- It can be beneficial to replace *`Conv2D`* layers with *`SeparableConv2D`* layers, which are more parameter efficient.

#### Exploration Task 3

<div class="alert alert-block alert-info"> 
<b>INSTRUCTIONS</b>
<br>
Perform the following operations:
<br>1. Altering Conv2D layer parameters
<br>1.1. stride > 1
<br>1.2. padding > 0
<br>1.3. dilation > 3
<br>2. Adding a maxpooling layer 
<br>3. Adding a batch normalization layer
<br>4. Adding a dropout layer
<br>5. Adding a dense layer
<br>6. Using a SeparableConv2D layer instead of Conv2D
<br>
<br>
<b>GUIDE QUESTIONS</b>
<li> What happens when stride, padding or dilation of the Conv2d layer are not set to default?
<li> For each layer/operation, note what happens to the shape and the number of parameters.
</div>

## Model Development

**Major Steps in Model Development**
> 1. Prepare the data.
> 2. Choose a model evaluation protocol.
> 3. Set a baseline.
> 4. Build the initial model.
> 5. Identify the `loss function`, `optimizer` and `metrics` appropriate to the use case.
> 6. Configure the learning process using the `compile()` method.
> 7. Identify the `number of epochs` and the `batch size`.
> 7. Set `callbacks` to perform desired actions at various stages of training (like saving a model or interrupting the training).
> 8. Implement the training loop using `fit()` method.
> 9. Regularize and tune your model until the best possible generalization performance is achieved.

#### Exploration Task 4

<div class="alert alert-block alert-info"> 
<b>INSTRUCTION</b>
<br>
Create an initial model to classify handwritten digits using the MNIST dataset.
</div>

**Step 1. Instantiate a small convnet**

In [ ]:
# Instantiate a small CNN model

In [ ]:
# Display the architecture of the model
model.summary()

You can see that the output of every Conv2D and MaxPooling2D layer is a 3D tensor of shape (height, width, channels). The width and height dimensions tend to shrink as you go deeper in the network. The number of channels is controlled by the first argument passed to the Conv2D layers (32 or 64).

Note that the default padding is zero $p=0$ and default stride is 1 $s=1$.

The 320 parameters in the first layer is because each image is convolved to a kernel/filter (weight) equal to (3 $\times$ 3) using 32 feature maps (3 $\times$ 3 $\times$ 32 = 9 $\times$ 32 = 288). Add a bias node for each of kernel in the 32 feature maps (1 $\times$ 32=32), and the total parameters is 288 + 32 = 320.

Max pooling will not add any new parameter but passing it into another 3 $\times$ 3 filter this time producing 64 feature maps for every 32 feature maps previously produce we have a total of  (32 $\times$ 64  $\times$ 3  $\times$ 3 =18,432) parameters. Add to this the bias for each of the 64 feature maps and the total is (18,432 + 64) 18,496. The process continues for the suceeding layers.

**Step 2. Add a classifier on top of the convnet**

The next step is to feed the last output tensor (of shape (3, 3, 64)) into a densely connected classifier network like those you’re already familiar with: a stack of Dense layers. These classifiers process vectors, which are 1D, whereas the current output is a 3D tensor. First we have to flatten the 3D outputs to 1D, and then add a few Dense layers on top.

In [ ]:
# Add a classifier on top

We’ll do 10-way classification, using a final layer with 1 output and a softmax activation.
Here’s what the network looks like now:

In [ ]:
model.summary()

As you can see above, the (3, 3, 64) outputs are flattened into vectors of shape (576,) before going through two Dense layers. The total parameters if we use a 64 hidden nodes on the flattened 576 features is (64 $\times$ 576 = 36,864) plus a built in biased in each of the 64 hidden nodes we get 36,928. Another small layer is added that contributes (64 $\times$ 1 + 1 = 65) parameters.

**Step 3. Train the convnet on the MNIST images**

Now, let’s train the convnet on the MNIST digits.

In [ ]:
# Perform the model completion step

In [ ]:
# Perform model fitting

**Step 4. Evaluate the model on the test data.**

In [ ]:
# Evaluate on test

In [ ]:
# Show sample output

<div class="alert alert-block alert-info"> 
Is this initial implementation correct?
</div>

## Exercise 3: Handwritten Digit Classification

**Instructions**
  
1. Improve the initial model until you obtain your best model. Consider tuning key gradient descent parameters, tuning the representational power of your model by chaning the number and/or type of layers or the size of the layers, or applying regularizations.

2. Report the model performances on the training, validation and test sets of your initial and best models. Describe the steps you undertook to improve the performance of your initial model.

3. Generate your own set of handwritten number images from 0 to 9. Test your best model on this set, and report its performance.

**Please use the exercise notebook to accomplish this task.**

## References

- Chollet, F., & Chollet, F. (2021). Deep learning with Python. simon and schuster.
- Bishop, C. M., & Bishop, H. (2023). Deep learning: Foundations and concepts. Springer Nature.
- Goodfellow, I., Bengio, Y., Courville, A., & Bengio, Y. (2016). Deep learning (Vol. 1, No. 2, pp. 1-800). Cambridge: MIT press.
- LeCun et al. (1989). Backpropagation Applied to Handwritten Zip Code Recognition. *Neural Computation* 1(4), pp. 541-551. doi: 10.1162/neco.1989.1.4.541.
- Monterola, CM. ML3 Notebook 2A.
- Simon, J. D. (2024). Understanding Deep Learning.

## Other Useful Resources

1. <a href="https://hannibunny.github.io/mlbook/neuralnetworks/03ConvolutionNeuralNetworks.html">Convolutional Neural Networks</a> by Johannes Maucher<br>
2. <a href="https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks">Stanford's CS230 CNN cheat sheet</a><br>
<br><b>ANIMATIONS</b><br>
- <a href="https://hannibunny.github.io/mlbook/neuralnetworks/convolutionDemos.html">Johannes Maucher's Convolution Demos</a>

<img src="images/banner-down.png" style="width: 100%;">